In [0]:
%run /Workspace/Users/rahulpatel@cyntexa.com/IAP_Practice/TPC-A/00_setup_t1

In [0]:
catalog = dbutils.widgets.get("catalog")

In [0]:
# ============================================================
# Data Quality Checks - TPC-A pipeline
# Each check appends to dq_results; the final cell reports
# the overall PASS / FAIL status.
# ============================================================

from pyspark.sql import functions as F

dq_results = []

def add_check(check_id, description, passed, details):
    status = "PASS" if passed else "FAIL"
    dq_results.append((check_id, description, status, str(details)))
    print(f"[{status}] {check_id} - {description}: {details}")

# ---- Check 1: all 8 tables ingested to bronze with data ----
bronze_tables = ["airline", "airline_route", "booking", "flight_booking", "nation", "passenger", "region", "route"]

bronze_counts = {t: spark.table(f"{catalog}.bronze.{t}").count() for t in bronze_tables}
empty_tables = [t for t, c in bronze_counts.items() if c == 0]

add_check(
    "DQ-01",
    "All 8 tables ingested to bronze with rows",
    len(bronze_counts) == 8 and not empty_tables,
    {"empty_tables": empty_tables} if empty_tables else {"all_tables_have_rows": True},
)

display(spark.createDataFrame(
    [(t, c) for t, c in bronze_counts.items()],
    ["bronze_table", "row_count"],
))

In [0]:
# ---- Check 2: primary keys populated and unique (silver layer) ----
pk_map = {
    "airline":        ["al_airlineid"],
    "airline_route":  ["ar_airlineid", "ar_routeid"],
    "booking":        ["bk_bookingid"],
    "flight_booking": ["fb_bookingid", "fb_linenumber"],
    "nation":         ["n_nationkey"],
    "passenger":      ["pa_passengerid"],
    "region":         ["r_regionkey"],
    "route":          ["ro_routeid"],
}

pk_rows = []
for table, keys in pk_map.items():
    tbl = f"{catalog}.silver.{table}"
    total = spark.table(tbl).count()
    null_keys = spark.table(tbl).where(" OR ".join(f"{k} IS NULL" for k in keys)).count()
    distinct_keys = spark.table(tbl).agg(F.countDistinct(*keys)).collect()[0][0]
    passed = (null_keys == 0) and (distinct_keys == total)
    pk_rows.append((table, ", ".join(keys), total, int(distinct_keys), null_keys, "PASS" if passed else "FAIL"))

add_check(
    "DQ-02",
    "Primary keys populated and unique in silver",
    all(r[5] == "PASS" for r in pk_rows),
    {r[0]: {"null_keys": r[4], "distinct": r[3], "rows": r[2]} for r in pk_rows},
)

display(spark.createDataFrame(
    pk_rows,
    ["table", "primary_key", "rows", "distinct_keys", "null_keys", "status"],
))

In [0]:
# ---- Check 3: fact table contains no null keys ----
fact_tbl = f"{catalog}.gold.fact_bookings"
fact_keys = ["booking_id", "line_number", "passenger_id", "route_id", "airline_id"]

null_key_counts = {k: spark.table(fact_tbl).where(F.col(k).isNull()).count() for k in fact_keys}

add_check(
    "DQ-03",
    "fact_bookings contains no null keys",
    all(v == 0 for v in null_key_counts.values()),
    null_key_counts,
)

# ---- Check 4: numeric measures satisfy business rules ----
violations = spark.sql(f"""
SELECT
    SUM(CASE WHEN ticket_price <= 0 THEN 1 ELSE 0 END)                       AS bad_ticket_price,
    SUM(CASE WHEN discount < 0 OR discount > 1 THEN 1 ELSE 0 END)            AS bad_discount,
    SUM(CASE WHEN seats < 1 THEN 1 ELSE 0 END)                                AS bad_seats
FROM {fact_tbl}
""").collect()[0]

rule_violations = {
    "ticket_price > 0":        int(violations["bad_ticket_price"]),
    "discount between 0 and 1": int(violations["bad_discount"]),
    "seats >= 1":               int(violations["bad_seats"]),
}

add_check(
    "DQ-04",
    "Numeric measures satisfy business rules",
    all(v == 0 for v in rule_violations.values()),
    rule_violations,
)

In [0]:
# ---- Check 5: gold revenue reconciles with silver (tolerance <= 0.01%) ----
gold_revenue = spark.sql(f"""
SELECT SUM(ticket_price * (1 - discount)) AS revenue
FROM {catalog}.gold.fact_bookings
""").collect()[0]["revenue"]

silver_revenue = spark.sql(f"""
SELECT SUM(CAST(fb_ticketprice AS DOUBLE) * (1 - CAST(fb_discount AS DOUBLE))) AS revenue
FROM {catalog}.silver.flight_booking
""").collect()[0]["revenue"]

abs_diff = abs(gold_revenue - silver_revenue)
pct_diff = (abs_diff / silver_revenue) * 100

add_check(
    "DQ-05",
    f"Gold revenue reconciles with silver ({pct_diff:.8f}% diff <= 0.01% tolerance)",
    pct_diff <= 0.01,
    {"gold_revenue": round(gold_revenue, 2), "silver_revenue": round(silver_revenue, 2), "pct_diff": round(pct_diff, 8)},
)

In [0]:
# ---- Check 6: rejected record counts per table (quarantine) ----
quarantine_rows = []
for table in ["airline", "airline_route", "booking", "flight_booking", "nation", "passenger", "region", "route"]:
    q_tbl = f"{catalog}.silver.{table}_quarantine"
    quarantine_rows.append((table, spark.table(q_tbl).count()))

add_check(
    "DQ-06",
    "Rejected record counts reported for all tables",
    len(quarantine_rows) == 8,
    {t: c for t, c in quarantine_rows},
)

display(spark.createDataFrame(
    quarantine_rows,
    ["table", "quarantined_rows"],
))

In [0]:
# ============================================================
# Overall Data Quality Status
# If ANY check fails, raise an error so the notebook task fails
# and the associated job run is stopped.
# ============================================================
overall = "PASS" if all(r[2] == "PASS" for r in dq_results) else "FAIL"

display(spark.createDataFrame(
    dq_results,
    ["check_id", "description", "status", "details"],
))

print("=" * 60)
print(f"OVERALL DATA QUALITY STATUS: {overall}")
print("=" * 60)

if overall == "FAIL":
    failed_checks = [f"{r[0]} ({r[1]})" for r in dq_results if r[2] == "FAIL"]
    raise RuntimeError(
        "Data quality validation failed - stopping the job run. "
        f"Failed checks: {'; '.join(failed_checks)}"
    )